# 04 — Basic Dataset Exploration (Groups, Users, Messages)

**Pipeline step:** Section 3.1 of the paper — descriptive statistics over the full raw
Telegram dataset (Blas et al., 2025), used to characterize chats, users, and messages before
any YouTube-link extraction begins.

**Purpose.** Load the full raw dataset with PySpark (too large for pandas) and report:
1. Groups vs. channels — average message length and most active chats/channels.
2. Users — total unique users, most active users, and how many distinct groups each user
   posts in.
3. Messages — overall average length (characters and words).
4. Timestamps — message volume per month.

**Input:**
- `../data/telegram_2024/extracted/*/2024-*.tsv.gz` — raw per-chat, per-month message files.

**Output:**
- `../data/User_message_counts.csv` — message count per user.
- `../data/User_group_counts.csv` — number of distinct groups each user posted in.

**Requires:** a running Spark session (`spark`). If you are not running this in an environment
that already provides one (e.g. Databricks), run the Spark session cell below first.

**Next step:** `03_Social_Media_Links_Study`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    input_file_name,
    max as spark_max,
    regexp_extract,
    sum as spark_sum,
)
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

In [ ]:
# If a `spark` session already exists in your environment (e.g. Databricks, EMR notebook),
# this cell is a no-op. Otherwise it starts a local Spark session.
spark = SparkSession.builder.appName("TelegramBasicAnalysis").getOrCreate()

In [ ]:
RAW_FILES_GLOB = "../data/telegram_2024/extracted/*/2024-*.tsv.gz"
OUTPUT_DIR = Path("../data")

# Schema of the raw per-message .tsv.gz files.
MESSAGE_SCHEMA = StructType([
    StructField("id", IntegerType(), False),
    StructField("user_id", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("bot_flag", BooleanType(), True),
    StructField("via_bot_id", BooleanType(), True),
    StructField("via_business_bot_id", BooleanType(), True),
    StructField("reply_to_msg_id", IntegerType(), True),
    StructField("fwd_flag", BooleanType(), True),
    StructField("fwd_from_id", IntegerType(), True),
    StructField("media_type", StringType(), True),
    StructField("views", IntegerType(), True),
    StructField("forwards", IntegerType(), True),
    StructField("replies", IntegerType(), True),
    StructField("reactions", IntegerType(), True),
    StructField("reaction_json", StringType(), True),
])

## Load and clean data

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("delimiter", "\t")
    .option("quote", '"')
    .option("multiline", True)
    .option("escape", '"')
    .schema(MESSAGE_SCHEMA)
    .csv(RAW_FILES_GLOB)
    .withColumn("file_path", input_file_name())
    # The chat/channel name is embedded in the file path, e.g. .../extracted/<group_name>/2024-03.tsv.gz
    .withColumn("group_name", regexp_extract(col("file_path"), r"/extracted/([^/]+)/", 1))
)

# Drop messages missing essential fields.
df = df.na.drop(subset=["id", "text", "timestamp"])

df.dtypes

In [ ]:
# Reused across the queries below.
df.select("user_id", "id", "text", "group_name", "timestamp").createOrReplaceTempView("messages")

print(f"Total messages after cleaning: {df.count()}")

## 1. Groups Analysis

Chat folders are named so that groups contain `"chat"` and channels contain `"channel"`
(see `01_Telegram_dataset_analysis.ipynb`).

In [ ]:
for label, pattern in [("chats", "%chat%"), ("channels", "%channel%")]:
    mean_chars = spark.sql(
        f"SELECT avg(LENGTH(text)) AS v FROM messages WHERE group_name LIKE '{pattern}'"
    ).first()["v"]
    mean_words = spark.sql(
        f"SELECT avg(SIZE(SPLIT(text, ' '))) AS v FROM messages WHERE group_name LIKE '{pattern}'"
    ).first()["v"]
    print(f"Average text length in {label}: {mean_chars:.2f} characters, {mean_words:.2f} words")

In [ ]:
print("Most active chats:")
df.filter(df.group_name.contains("chat")).groupBy("group_name").count() \
  .orderBy("count", ascending=False).show()

print("Most active channels:")
df.filter(df.group_name.contains("channel")).groupBy("group_name").count() \
  .orderBy("count", ascending=False).show()

## 2. Users Analysis

In [ ]:
total_users = df.select("user_id").distinct().count()
print(f"Total unique users: {total_users}")

df_user_message_counts = df.groupBy("user_id").count()
print("Most active users:")
df_user_message_counts.orderBy("count", ascending=False).show()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_user_message_counts.toPandas().to_csv(OUTPUT_DIR / "User_message_counts.csv", index=False)

In [ ]:
print("Number of distinct groups each user posted in:")
user_group_counts = spark.sql(
    "SELECT user_id, COUNT(DISTINCT group_name) AS group_count "
    "FROM messages GROUP BY user_id ORDER BY group_count DESC"
)
user_group_counts.toPandas().to_csv(OUTPUT_DIR / "User_group_counts.csv", index=False)
user_group_counts.show()

In [ ]:
summary = df_user_message_counts.agg(
    avg("count").alias("avg_messages_per_user"),
    spark_sum("count").alias("total_messages_in_dataset"),
    spark_max("count").alias("max_messages_by_a_single_user"),
    count("user_id").alias("num_unique_users"),
)
summary.show()

## 3. Message Analysis

In [ ]:
mean_chars = spark.sql("SELECT avg(LENGTH(text)) AS v FROM messages").first()["v"]
mean_words = spark.sql("SELECT avg(SIZE(SPLIT(text, ' '))) AS v FROM messages").first()["v"]

print(f"Average text length (overall): {mean_chars:.2f} characters, {mean_words:.2f} words")

## 4. Timestamp Analysis

In [ ]:
messages_per_month = spark.sql("""
    SELECT
        DATE_FORMAT(FROM_UNIXTIME(timestamp), 'yyyy-MM') AS month,
        COUNT(*) AS count_messages
    FROM messages
    GROUP BY DATE_FORMAT(FROM_UNIXTIME(timestamp), 'yyyy-MM')
    ORDER BY count_messages DESC
""")

messages_per_month.show()